# FSOT Biohub v50 — competitive U-Net + FSOT vision

Pipeline: **U-Net detection** (FSOT-calibrated threshold) → **FSOT fsot_gate linking** → `submission.csv`

**Inputs:** competition test + `cellmot-ft-detector-biohub` + `cellmot-baseline-artifacts` + `fsot-v50-competitive-bundle`

Lean ref: https://github.com/dappalumbo91/FSOT-2.1-Lean

In [ ]:
import os
import shutil
import glob
import subprocess
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("BIOHUB_ENGINE", "auto")
os.environ.setdefault("FSOT_VISION_CALIBRATE", "1")
os.environ.setdefault("FSOT_LINK_MODE", "fsot_gate")
os.environ.setdefault("FSOT_GATE_FRAC", "0.42")
os.environ.setdefault("FSOT_GATE_ADAPTIVE", "1")
os.environ.setdefault("FSOT_GATE_RESCUE", "1")
os.environ.setdefault("CELLMOT_USE_ILP", "0")
os.environ.setdefault("CELLMOT_DET_THRESHOLD", "0.45")
os.environ.setdefault("CELLMOT_NMS_UM", "6.0")
os.environ.setdefault("FSOT_LIVING_EMERGENCE", "0")
os.environ.setdefault("CELLMOT_EDGE_THRESHOLD", "0.25")
os.environ.setdefault("CELLMOT_USE_FT", "1")
os.environ.setdefault("FSOT_GAP_LINK", "1")
os.environ.setdefault("KAGGLE_SUBMISSION_FAST_VALIDATE", "0")

print("Kaggle input:", os.listdir("/kaggle/input"))

_bundle_hits = glob.glob("/kaggle/input/**/kaggle_main_runner.py", recursive=True)
if _bundle_hits:
    src_dir = Path(_bundle_hits[0]).parent
    for name in [
        "kaggle_main_runner.py",
        "fsot_vision_calibrate.py",
        "fsot_living_emergence.py",
        "download_ft_weights.py",
        "fsot_original_competition.py",
        "fsot_core.py",
        "fsot_cellular_bridge.py",
        "biohub_competitive.py",
        "biohub_unet_engine.py",
        "submission_io.py",
        "validate_kaggle_submission.py",
        "csv_to_geffs.py",
    ]:
        p = src_dir / name
        if p.exists():
            shutil.copy2(p, WORK / name)
    print(f"Copied v50 bundle from {src_dir}")
else:
    print("WARN: fsot-v50-competitive-bundle not found")

_ft = glob.glob("/kaggle/input/**/cellmot-ft-detector-biohub/**/edge_predictor_best.pth", recursive=True)
_wt = _ft or glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True)
if _wt:
    os.environ["CELLMOT_UNET_WEIGHTS"] = _wt[0]
    print(f"[UNET] weights: {_wt[0]}")

for bundle_path in glob.glob("/kaggle/input/**/cellmot_code_bundle.zip", recursive=True):
    with zipfile.ZipFile(bundle_path, "r") as zf:
        zf.extractall("/kaggle/working")
    print(f"[CELLMOT] extracted {bundle_path}")
    break

_wheels = glob.glob("/kaggle/input/**/tracksdata*.whl", recursive=True)
if _wheels:
    wheel_dir = os.path.dirname(_wheels[0])
    subprocess.run(
        f"pip install --no-index --find-links {wheel_dir} tracksdata geff zarr dask polars",
        shell=True, capture_output=True, text=True,
    )
    print(f"Offline wheels: {wheel_dir}")

In [ ]:
import runpy
import sys

sys.path.insert(0, "/kaggle/working")
runpy.run_path("/kaggle/working/kaggle_main_runner.py", run_name="__main__")

import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.groupby(["dataset", "row_type"]).size())
print(f"submission rows: {len(sub)}")